# 04 · Jev-as-IF：PyIF 程序、静态检查与 Jev 原语

1. Jev 连通性。
2. **PyIF 静态检查**：用 SOG 检查 agent 写的程序（不花钱）。
3. **执行手写程序**：工具走 MCP 回放，语义判断走 Jev，看轨迹和每个判断的概率、门控结果。
4. **问题构造对比（消融 B3 的预演）**：同样的判断题，只给原始代码 / 加标签 / 完整构造。
5. **门控标定**：在 FinMCP 抽出的判断题上测 Jev 的准确率，用共形方法算阈值。

**会调用 Jev（每次几百到几千 token，按 $0.042/百万 token 计费，整本远不到 1 美分）**；不调用大模型。

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd
from jif import config, finmcp, build, calib, pyif, sog as S
from jif.jevrt import JevRuntime, Gate, conformal_tau
from jif.llm import Ledger
from jif.mcp_conn import McpConnection
pd.set_option("display.max_colwidth", 120)

keys = config.load_keys()
print(config.describe(keys))
path = config.ARTIFACTS / "sog_finmcp.json"
if path.exists():
    sog = S.SOG.load(path)
else:
    print("没有找到 01 号 notebook 保存的 SOG，现在自动建一张（未经人工整理）")
    sog, _ = build.build_sog(finmcp.iter_tasks(("multi_tool", "multi_turn")))
ops = sog.ops()
print(sog.label, "：", len(sog.objects.entities), "个实体，", len(ops), "个操作")

## 1. Jev 连通性

In [ ]:
from typesafe_sdk import TypeSafeClient, Choice, Noul
client = TypeSafeClient(model=keys["JEV_MODEL"])
r = client.system_one("我想把稳健优选那个组合赎回，手续费多少？", {
    "intent": Choice(instructions="客户这句话的主要意图", criteria={"redeem": "赎回", "fee": "询问费用", "other": None}),
    "mentions_fee": Noul(instructions="客户是否问到了费用？")})
print(r.model, r.usage.input_tokens, "tokens")
print(r.choices["intent"].choice, r.choices["intent"].probabilities, "| mentions_fee =", r.nouls["mentions_fee"].noul)

## 2. PyIF 静态检查

对应论文 §4.3。下面一段合法程序，加上几段典型的错误程序。检查完全在本地完成。

In [ ]:
task30 = finmcp.get_task("multi_tool:30")
q30 = finmcp.first_question(task30)
GOOD = """a = StrategySearchByKeyword(keyword="长赢")
b = StrategySearchByKeyword(keyword="稳稳的幸福")
picked = jev.choice("客户问到的是哪些策略？", a.rows + b.rows, k="*")
dist = GetUserAssetHoldingDistribution(poCodes=[s.策略代码 for s in picked])
answer(strategies=picked, holdings=dist)"""
CASES = {
    "合法程序": GOOD,
    "猜错字段名": GOOD.replace("s.策略代码", "s.strategyCode"),
    "传了身份参数": GOOD.replace("GetUserAssetHoldingDistribution(", 'GetUserAssetHoldingDistribution(userId="123", '),
    "写死策略代码": GOOD.replace("[s.策略代码 for s in picked]", '["ZH999999"]'),
    "用位置参数": GOOD.replace('keyword="长赢"', '"长赢"'),
    "候选是写死的列表": GOOD.replace("a.rows + b.rows", '["长赢", "稳稳的幸福"]'),
    "不允许的语法": "import os\n" + GOOD,
    "缺少 answer": GOOD.rsplit("\n", 1)[0],
}
pd.DataFrame([{"程序": k, "问题": "；".join(map(str, pyif.check(code, ops, q30))) or "通过"} for k, code in CASES.items()])

## 3. 执行手写程序（MCP 回放 + Jev）

In [ ]:
ledger = Ledger()
conn = McpConnection()
jev = JevRuntime(sog, model=keys["JEV_MODEL"], ledger=ledger)    # 没有配置回退：不确定时用 Jev 的最佳猜测并标为 ungated

def run_program(tid, code):
    task = finmcp.get_task(tid)
    conn.set_task(tid)
    q = finmcp.first_question(task)
    issues = pyif.check(code, ops, q)
    if issues:
        print("静态检查未通过：", issues)
        return None
    res = pyif.Interpreter(ops, conn.call, jev, request=q, identity={"userId": finmcp.user_id(task)}).run(code)
    print("问题：", q)
    print("错误：", res.error)
    for t in res.trace:
        if t["type"] == "tool":
            print(f"  工具 {t['name']}({json.dumps(t['args'], ensure_ascii=False)[:100]}) ok={t['ok']}")
    print("  回放：", [e["status"] for e in conn.replay_log()])
    display(pd.DataFrame([{"原语": d.kind, "问题": d.question, "候选数": len(d.options), "门控": d.gated,
                           "结果": d.result, "最高的几个概率": dict(sorted(d.probs.items(), key=lambda kv: -kv[1])[:4])}
                          for d in jev.decisions]))
    return res

res30 = run_program("multi_tool:30", GOOD)

In [ ]:
P1 = """r = StrategySearchByKeyword(keyword="海外", pageNum=1, pageSize=20)
overseas = jev.filter("这个策略是否主要投资海外市场？", r.rows)
d = GetStrategyDetails(strategyCodes=[s.策略代码 for s in overseas], pageNum=1, pageSize=3)
answer(strategies=d.rows)"""
res1 = run_program("multi_tool:1", P1)

## 4. 问题构造对比：原始代码 / 加标签 / 完整构造

从 FinMCP 轨迹里抽出"从多个候选中选一个"的判断题（状态 = 客户原话，候选 = 上游返回的记录，答案 = 标准轨迹的选择），分别用三种方式构造 Jev 的问题。这是实验设计里消融 B3 的预演。

In [ ]:
N_ITEMS = 30
items = calib.choice_items(("multi_tool", "multi_turn"), limit=N_ITEMS)
print("抽到判断题：", len(items), "道；候选数中位数：", pd.Series([i["n"] for i in items]).median())

def evaluate(mode, items):
    rt = JevRuntime(sog, model=keys["JEV_MODEL"], elaboration=mode, gate=Gate(enabled=False), ledger=ledger)
    out = []
    for it in items:
        rt.start(it["request"])
        rt.observe(it["items"])
        pick = rt.choice("客户在请求里指的是下面哪一个？", it["items"], key_field=it["id_key"])
        d = rt.decisions[-1]
        out.append({"mode": mode, "task": it["task_id"], "n": it["n"], "gold": it["gold"], "pick": d.result,
                    "correct": d.result == it["gold"], "p_gold": d.probs.get(it["gold"], 0.0),
                    "p_top": max(d.probs.values())})
    return out

ablation = pd.DataFrame([row for mode in ("raw", "label", "full") for row in evaluate(mode, items)])
ablation.groupby("mode")[["correct", "p_gold", "p_top"]].mean().round(3)

## 5. 门控标定（共形）

用完整构造的结果，按论文 §4.6 的方法算阈值：标定集上的分数是 `1 - p(正确选项)`，取 `(n+1)(1-α)` 分位数作为阈值 τ；运行时只有预测集是单元素时才执行。样本很少时结果只是示意。

In [ ]:
full = ablation[ablation["mode"] == "full"].reset_index(drop=True)
half = len(full) // 2
cal, test = full.iloc[:half], full.iloc[half:]
rows = []
for alpha in (0.1, 0.2, 0.3):
    tau = conformal_tau(list(cal["p_gold"]), alpha)
    act = test["p_top"] >= 1 - tau              # 单元素预测集的近似判据（忽略第二名也进入集合的情况）
    rows.append({"α": alpha, "τ": round(tau, 3), "执行比例": round(act.mean(), 3),
                 "执行且判错的比例": round((act & ~test["correct"]).mean(), 3),
                 "执行部分的准确率": round(test[act]["correct"].mean(), 3) if act.any() else None})
display(pd.DataFrame(rows))
print("测试集上不设门控的准确率：", round(test["correct"].mean(), 3))

In [ ]:
display(pd.DataFrame(Ledger.summarize(ledger.records)).T)
conn.close()